# Домашнее задание: прогноз экономического ряда

**Ряд:** среднемесячная спотовая цена нефти WTI, USD/баррель  
**Источник:** FRED, серия [WTISPLC](https://fred.stlouisfed.org/series/WTISPLC)  
**Период:** январь 2010 — август 2026 (200 месяцев)  
**Hold-out:** последние 10 месяцев, ноябрь 2025 — август 2026. Эти точки **не участвуют** в оценке параметров и подборе порядка.

Ряды с занятий не используем: влажность городов, MSFT из лекций 5–6, учебный `practice_series`.

**Показатель качества out-of-sample:** RMSE (рядом смотрим MAE). RMSE сильнее штрафует большой промах на одном месяце — для цены нефти это уместно.

Регрессоры тоже с FRED: INDPRO (промпроизводство США), DTWEXBGS (широкий индекс доллара). Зимний dummy строится из календаря.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA

try:
    import pmdarima as pm
    HAS_PM = True
except ImportError:
    HAS_PM = False
    print('pmdarima нет — auto-ARIMA пропустим, порядок возьмём сеткой')

plt.rcParams['figure.figsize'] = (10, 3.6)


def metrics(y_true, y_hat, tag):
    y_true = np.asarray(y_true, float)
    y_hat = np.asarray(y_hat, float)
    mae = np.mean(np.abs(y_true - y_hat))
    rmse = np.sqrt(np.mean((y_true - y_hat) ** 2))
    print(f'{tag:24s}  MAE={mae:6.2f}  RMSE={rmse:6.2f}')
    return mae, rmse


## 1. Данные, визуализация, особенности ряда


In [ ]:
def fred_month(series_id, name):
    df = pd.read_csv(f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}')
    df.columns = ['date', name]
    df['date'] = pd.to_datetime(df['date'])
    s = pd.to_numeric(df.set_index('date')[name], errors='coerce')
    if series_id == 'DTWEXBGS':
        s = s.resample('MS').mean()
    else:
        s = s.asfreq('MS')
    return s


try:
    df = pd.read_csv('wti_hw_data.csv', parse_dates=['date'], index_col='date')
    print('взяли локальный wti_hw_data.csv')
except FileNotFoundError:
    wti = fred_month('WTISPLC', 'wti')
    indpro = fred_month('INDPRO', 'indpro')
    dollar = fred_month('DTWEXBGS', 'dollar')
    df = pd.concat([wti, indpro, dollar], axis=1).loc['2010-01-01':'2026-08-01']
    df['winter'] = df.index.month.isin([11, 12, 1, 2, 3]).astype(int)
    df = df.dropna()

if 'winter' not in df.columns:
    df['winter'] = df.index.month.isin([11, 12, 1, 2, 3]).astype(int)

print(df.shape)
print(df.head())
print(df.tail())


In [ ]:
y = df['wti']
train, test = y.iloc[:-10], y.iloc[-10:]
exog = df[['indpro', 'dollar', 'winter']]
ex_tr, ex_te = exog.iloc[:-10], exog.iloc[-10:]

print('train:', train.index.min().date(), '→', train.index.max().date(), 'n =', len(train))
print('последнее train:', round(float(train.iloc[-1]), 2))
print('test :', [d.strftime('%Y-%m') for d in test.index])
print(test.round(2))


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(train.index, train, color='#1a5276', label='train')
ax.plot(test.index, test, color='#c0392b', label='test (10 мес., не в обучении)')
ax.axvline(test.index[0], color='gray', ls='--', lw=0.8)
ax.set_title('WTI, USD/баррель, месяц')
ax.set_ylabel('USD')
ax.legend()
plt.tight_layout()
plt.show()

print(train.describe().round(2))


**Что видно по графику**

- Уровень цены не постоянный: обвалы 2014 и 2020, скачок 2022.
- Это стохастический тренд / почти случайное блуждание, а не пила вокруг одного среднего.
- Календарная сезонность слабая: нефть не температура. Зимний спрос есть, его перекрывают шоки.
- На test (ноя 2025 — авг 2026) цена сначала ~58–64, в марте–мае 2026 уходит к 91–102. Модель, обученная до октября 2025, этот скачок не «знает».


## 2. Стационарность (2 способа) и автокорреляция (2 способа)


In [ ]:
def stationarity_report(s, name):
    s = s.dropna()
    adf = adfuller(s, autolag='AIC')
    kps = kpss(s, regression='c', nlags='auto')
    print(f'=== {name} ===')
    print(f'ADF  stat={adf[0]:.3f}  p={adf[1]:.4f}   H0: нестационарен')
    print(f'KPSS stat={kps[0]:.3f}  p={kps[1]:.4f}   H0: стационарен')
    if adf[1] <= 0.05 and kps[1] > 0.05:
        print('вердикт: стационарен (оба согласны)')
    elif adf[1] > 0.05 and kps[1] <= 0.05:
        print('вердикт: нестационарен (оба согласны)')
    else:
        print('вердикт: тесты спорят — смотрим график и преобразование')
    print()


stationarity_report(train, 'уровень WTI, train')
stationarity_report(train.diff(), 'первая разность, train')
stationarity_report(train.pct_change(), 'процентный прирост, train')


In [ ]:
print('Ljung-Box уровня')
print(acorr_ljungbox(train, lags=[6, 12, 24], return_df=True))
print('\nLjung-Box первой разности')
print(acorr_ljungbox(train.diff().dropna(), lags=[6, 12, 24], return_df=True))

fig, ax = plt.subplots(2, 2, figsize=(11, 6))
plot_acf(train, ax=ax[0, 0], lags=24)
ax[0, 0].set_title('ACF уровня')
plot_pacf(train, ax=ax[0, 1], lags=24)
ax[0, 1].set_title('PACF уровня')
plot_acf(train.diff().dropna(), ax=ax[1, 0], lags=24)
ax[1, 0].set_title('ACF разности')
plot_pacf(train.diff().dropna(), ax=ax[1, 1], lags=24)
ax[1, 1].set_title('PACF разности')
plt.tight_layout()
plt.show()


**Как читать**

| проверка | способ 1 | способ 2 |
|---|---|---|
| стационарность | ADF | KPSS |
| автокорреляция | Ljung-Box | ACF / PACF |

На прогоне (train до окт. 2025):

- уровень: ADF p ≈ **0.22**, KPSS p ≈ **0.07** — тесты **спорят** (случай C). График и медленный ACF уровня всё равно говорят про блуждание, не про постоянное среднее.
- разность: ADF p ≈ **0**, KPSS p ≈ **0.10** — оба за стационарность.
- Ljung-Box уровня p = 0 на 6/12/24. На разности лаг 6 p ≈ 0.01, лаг 12 p ≈ 0.11 — короткая память осталась, белый шум это ещё не.
- ACF уровня почти не тает. ACF разности обрыв после лага 1 → старт **MA(1)**.

Почему `d = 1`, а не «оставляем уровень»: ARMA требует слабой стационарности. Среднее WTI за 2010 и за 2022 разное.

Почему не проценты как рабочий ряд: проценты тоже стационарны, но задание про прогноз **цены в долларах**. Разность прямо даёт ARIMA на уровне.


## 3. ARMA на стационарном ряде (разности)

Учим ARMA(p,q) на Δy. Прогноз разности складываем с последним уровнем train. Это тот же смысл, что ARIMA(p,1,q) на цене.


In [ ]:
d_train = train.diff().dropna()

arma_grid = {}
for p in range(0, 3):
    for q in range(0, 3):
        m = ARIMA(d_train, order=(p, 0, q)).fit()
        arma_grid[(p, q)] = m.aic
        print(f'ARMA({p},{q}) на разности  AIC={m.aic:.1f}  BIC={m.bic:.1f}')

best_pq = min(arma_grid, key=arma_grid.get)
print('лучший по AIC на разности:', best_pq)


In [ ]:
arma = ARIMA(d_train, order=(best_pq[0], 0, best_pq[1])).fit()
print(arma.summary())
print(acorr_ljungbox(arma.resid.dropna(), lags=[6, 12], return_df=True))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
plot_acf(arma.resid.dropna(), ax=ax[0], lags=20)
ax[0].set_title('ACF остатков ARMA')
plot_pacf(arma.resid.dropna(), ax=ax[1], lags=20)
ax[1].set_title('PACF остатков ARMA')
plt.tight_layout()
plt.show()


In [ ]:
fc_d = arma.forecast(steps=len(test))
arma_oos_level = train.iloc[-1] + np.cumsum(np.asarray(fc_d))

print('ARMA на разности → восстановленный уровень')
metrics(test, arma_oos_level, 'ARMA OOS')


На прогоне лучший AIC у **ARMA(0,1)** (≈ 1185) и почти такой же у ARMA(1,2). Остатки тише сырой разности. Дальше на уровне это будет ARIMA(0,1,1).


## 4. ARIMA / auto-ARIMA на исходном уровне

`d = 1` уже из тестов. `p` и `q` — сетка и auto-ARIMA. `d` по AIC не выбираем: разный `d` меняет шкалу правдоподобия.


In [ ]:
if HAS_PM:
    auto = pm.auto_arima(
        train, seasonal=False, stepwise=True,
        suppress_warnings=True, error_action='ignore', trace=True,
    )
    print('auto-ARIMA order =', auto.order, 'AIC =', auto.aic())
    arima_order = auto.order
else:
    arima_order = (0, 1, 1)
    print('ручной порядок', arima_order)

print('\nсетка на уровне при d=1:')
for p in range(0, 3):
    for q in range(0, 3):
        m = ARIMA(train, order=(p, 1, q)).fit()
        print(f'ARIMA({p},1,{q})  AIC={m.aic:.1f}  BIC={m.bic:.1f}')


In [ ]:
def fit_eval(order, y_tr, y_te, ex_tr=None, ex_te=None, name=''):
    m = ARIMA(y_tr, order=order, exog=ex_tr).fit()
    ins = m.fittedvalues.reindex(y_tr.index)
    oos = m.forecast(steps=len(y_te), exog=ex_te)
    resid = m.resid.dropna()
    lb = acorr_ljungbox(resid, lags=[6, 12], return_df=True)
    print('\n', name, 'order=', order)
    print('AIC={:.1f}  BIC={:.1f}'.format(m.aic, m.bic))
    print('Ljung-Box остатков\n', lb)
    print('p-value коэффициентов:\n', m.pvalues.round(4))
    metrics(y_tr.iloc[1:], ins.iloc[1:], name + ' in-sample')
    metrics(y_te, oos, name + ' OOS')
    return m, ins, oos, resid


naive_oos = np.full(len(test), float(train.iloc[-1]))
print('наивный прогноз = последнее train =', round(float(train.iloc[-1]), 2))
metrics(test, naive_oos, 'naive OOS')

res_arima, is_arima, oos_arima, r_arima = fit_eval(
    arima_order, train, test, name='ARIMA'
)


In [ ]:
print(res_arima.summary())
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
plot_acf(r_arima, ax=ax[0], lags=20)
ax[0].set_title('ACF остатков ARIMA')
plot_pacf(r_arima, ax=ax[1], lags=20)
ax[1].set_title('PACF остатков ARIMA')
plt.tight_layout()
plt.show()


На прогоне ARIMA(0,1,1): AIC ≈ 1183, ma.L1 значим, Ljung-Box остатков на 6 лагах p ≈ 0.84 — короткая память снята. In-sample линия почти совпадает с фактом (случай I). OOS RMSE ≈ 24.5, наивный ≈ 23.9: чистая ARIMA **не бьёт** «завтра = последний месяц».


## 5. Регрессоры и ARIMAX

Нужны и dummy, и непрерывные, всего ≥ 3.

| регрессор | тип | зачем | откуда на будущее |
|---|---|---|---|
| `indpro` — промпроизводство США | непрерывный | спрос на топливо в деловом цикле | FRED INDPRO. На 10 hold-out берём **факт** (условный прогноз). В бою — прогноз INDPRO / nowcast |
| `dollar` — широкий индекс доллара | непрерывный | нефть в USD: крепкий доллар давит на цену | FRED DTWEXBGS, месяц. На hold-out — факт. В бою — прогноз курса |
| `winter` — ноябрь–март | dummy | отопительный сезон | календарь, известен на любую дату вперёд |

Dummy на test без утечки. Непрерывные на этих 10 месяцах — реализованные значения, чтобы сравнить модели при одной и той же информации об экзогенах. Целевую цену WTI из test модели не видят.


In [ ]:
print(ex_tr.tail())
print(ex_te)

res_x, is_x, oos_x, r_x = fit_eval(
    (0, 1, 1), train, test, ex_tr, ex_te, name='ARIMAX(0,1,1)'
)
print(res_x.summary())


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
plot_acf(r_x, ax=ax[0], lags=20)
ax[0].set_title('ACF остатков ARIMAX')
plot_pacf(r_x, ax=ax[1], lags=20)
ax[1].set_title('PACF остатков ARIMAX')
plt.tight_layout()
plt.show()
print(acorr_ljungbox(r_x, lags=[6, 12, 24], return_df=True))


На прогоне ARIMAX(0,1,1): AIC ≈ **1164** (лучше чистой ARIMA на ~20 пунктов). Значимы `indpro` (p ≈ 0.02) и `dollar` (p ≈ 0.00). `winter` p ≈ 0.27 — формально лишний, оставляем, потому что задание просит dummy. Остатки тихие (LB 6/12 p ≈ 0.99). OOS RMSE ≈ **22.2** против наивных 23.9.


## 6. Сравнение прогнозов с фактом


In [ ]:
cmp = pd.DataFrame({
    'fact': test.values,
    'naive': naive_oos,
    'ARMA_on_diff': np.asarray(arma_oos_level),
    'ARIMA': np.asarray(oos_arima),
    'ARIMAX': np.asarray(oos_x),
}, index=test.index)
print(cmp.round(2))

print('\n--- out-of-sample ---')
for col in ['naive', 'ARMA_on_diff', 'ARIMA', 'ARIMAX']:
    metrics(cmp['fact'], cmp[col], col)

fig, ax = plt.subplots(figsize=(11, 4.2))
ax.plot(train.index[-24:], train.iloc[-24:], color='#1a5276', label='train (хвост)')
ax.plot(test.index, test, color='black', lw=2, label='факт test')
ax.plot(test.index, cmp['naive'], ls=':', label='naive = последнее train')
ax.plot(test.index, cmp['ARIMA'], label='ARIMA')
ax.plot(test.index, cmp['ARIMAX'], label='ARIMAX')
ax.legend()
ax.set_title('Out-of-sample, 10 месяцев')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(train.index, train, color='silver', label='факт')
ax.plot(train.index, is_arima, color='#1a5276', alpha=0.85, label='ARIMA in-sample')
ax.plot(train.index, is_x, color='#c0392b', alpha=0.7, label='ARIMAX in-sample')
ax.set_title('In-sample (не критерий выбора модели)')
ax.legend()
plt.tight_layout()
plt.show()


## 7. Выводы и выбор модели

### Почему такая схема

- **Стационарность.** Уровень: ADF не отвергает корень, KPSS на 5% ещё «стационарен» (p ≈ 0.07) — спор. График, медленный ACF и экономика цены говорят брать одну разность. После `diff` оба теста согласны. Сезонную разность 12 не ставили: устойчивой календарной волны, сопоставимой с шоками, нет.
- **Порядок.** На разности ACF обрыв после 1 → MA(1) → **ARIMA(0,1,1)**. Сетка AIC это подтверждает. Соседи (1,1,0)/(1,1,1) почти те же, лишние члены часто незначимы.
- **Остатки.** У ARIMA и ARIMAX Ljung-Box 6–12 p ≫ 0.05: короткая память снята. Тяжёлые хвосты кризисов 2014/2020 порядком ARMA не лечатся.
- **Регрессоры.** Промпроизводство и доллар двигают спрос и номинальную цену. Зима — календарь. На hold-out непрерывные взяты как факт; в проде их нужно прогнозировать отдельно.
- **Метрика.** Главная — **RMSE на 10 out-of-sample месяцах**. In-sample у всех красивый (случай I) и модель не выбирает.
- **Наивный бенчмарк.** «Следующий месяц = последний известный». Если модель его не бьёт, сложность не оправдана.

### Цифры прогона (train до 2025-10, test 10 месяцев)

| модель | AIC | RMSE OOS | MAE OOS |
|---|---|---|---|
| naive (последний train = 60.89) | — | **23.86** | 18.59 |
| ARMA на разности / ARIMA(0,1,1) | ≈ 1183 | 24.5 | 19.0 |
| **ARIMAX(0,1,1)** + indpro, dollar, winter | ≈ **1164** | **22.2** | **17.2** |

Все модели на марте–мае 2026 остаются около 60–65 и не ловят скачок к 100. ARIMAX чуть ближе за счёт доллара и промпроизводства, не потому что «угадала шок».

### Какая модель лучшая и почему на этом ряде

По RMSE out-of-sample и при тихих остатках лучшая — **ARIMAX(0,1,1)** с тремя регрессорами.

Почему не уровень + ARMA: ряд нестационарен, ARMA на уровне строить нельзя.  
Почему не одна ARIMA: на этом hold-out она проигрывает наивной. Экзогены забирают кусок дисперсии, AIC падает, OOS чуть лучше.  
Почему победа скромная: на этих 10 месяцах главный сюжет — внешний скачок, а не MA(1). Для такого ряда «лучшая» в рамках задания — модель, которая честно описывает стационарные *приращения* и не проигрывает наивной на уровне.

Если FRED обновит последний месяц, числа в ячейках чуть поедут — в отчёт копируй вывод своих ячеек.
